## Executive Summary

**Purpose:** put every part together. For one winding, this gives the full phase inductance matrix and resistance, the synchronous and zero-sequence values, and the imbalance. For several candidate windings on the same stator, it gives a side-by-side table.

**What it does:** `stator_inductances(pattern, stator, end, ...)` returns a `StatorInductances`. It holds the magnetising, harmonic, slot, tooth-tip and end-winding matrices, with `total`, `leakage`, `positive()`, `zero()`, `imbalance()` and `summary()`. `compare_windings(patterns, stator, end, ...)` puts the key numbers of each winding in one table.

**Background:** what each part is and what a measurement contains: [Inductance Theory](inductance_theory.html).

**Assumptions:** those of the parts: linear iron, a smooth rotor, a 1-D slot field and an empirical end-winding factor.

## How It Fits Into the Motor Design Workflow

**Upstream:** [Air-Gap Inductance](inductance_airgap.html), [Slot and Tooth-Tip Leakage](inductance_slot.html), [End Turns, Phase Resistance and End-Winding Leakage](inductance_end_winding.html).

**Downstream:** motor models ($L_s = L_d = L_q$ for a smooth rotor, $R_s$), winding selection, and the saturated MEC model planned next.

## Theory

The total phase matrix is the sum of the parts, each computed with the same turns per slot position:

$$\mathbf{L} = \underbrace{\mathbf{L}_m + \mathbf{L}_\delta}_{\text{air gap}} + \mathbf{L}_{slot} + \mathbf{L}_{tt} + \mathbf{L}_{ew}, \qquad \mathbf{L}_\sigma = \mathbf{L} - \mathbf{L}_m.$$

For each part, `positive(part)` gives the positive-sequence (synchronous) value $L_{aa} - L_{ab}$ (exactly so for a balanced 3-phase winding), and `zero(part)` the zero-sequence value $\tfrac1m \sum_{xy} L_{xy}$. `imbalance()` is `phase_imbalance` of the total, plus the resistance spread.

![Stacked bars of the positive-sequence inductance of four windings on the same 48-slot stator, split into magnetising, harmonic, slot, tooth-tip and end-winding parts, with the self-inductance and resistance spread of each written above its bar.](images/winding_comparison.svg)

In [ ]:
#| default_exp inductance.stator

In [ ]:
#| hide
from emachines.inductance.geometry import SlotGeometry
from emachines.winding.pattern import Coil

In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from emachines.inductance.airgap import airgap_inductance, phase_imbalance
from emachines.inductance.endwinding import EndTurn, end_winding_inductance, phase_resistance
from emachines.inductance.geometry import StatorGeometry
from emachines.inductance.slot import slot_leakage_inductance
from emachines.winding.pattern import WindingPattern

In [ ]:
#| export
@dataclass(frozen=True)
class StatorInductances:
    """
    Every part of a winding's stator inductance, as m × m phase matrices [H], and its resistance.

    Build it with ``stator_inductances``.

    Attributes
    ----------
    magnetising : np.ndarray   Air gap, working harmonic ν = p
    harmonic    : np.ndarray   Air gap, all other harmonics (differential leakage)
    slot        : np.ndarray   Slot body, opening and tooth-tip transition
    tooth_tip   : np.ndarray   Tooth-tip leakage at the gap
    end_winding : np.ndarray   End-winding leakage (empirical, diagonal)
    resistance  : np.ndarray | None   DC resistance per phase [Ω], if computed
    name        : str
    """
    magnetising: np.ndarray = field(repr=False)
    harmonic: np.ndarray = field(repr=False)
    slot: np.ndarray = field(repr=False)
    tooth_tip: np.ndarray = field(repr=False)
    end_winding: np.ndarray = field(repr=False)
    resistance: np.ndarray | None = field(default=None, repr=False)
    name: str = ""

    @property
    def leakage(self) -> np.ndarray:
        """L_σ = harmonic + slot + tooth tip + end winding."""
        return self.harmonic + self.slot + self.tooth_tip + self.end_winding

    @property
    def total(self) -> np.ndarray:
        """The phase inductance matrix a terminal measurement sees (linear, smooth rotor)."""
        return self.magnetising + self.leakage

    def positive(self, part: str = "total") -> float:
        """Positive-sequence (synchronous) inductance of a part: 'total', 'leakage' or a field name."""
        return phase_imbalance(getattr(self, part))["L_positive"]

    def zero(self, part: str = "total") -> float:
        """Zero-sequence inductance (1/m) Σ L_xy of a part."""
        L = getattr(self, part)
        return float(L.sum() / L.shape[0])

    def imbalance(self) -> dict:
        """``phase_imbalance`` of the total, plus the resistance spread (max − min)/mean if known."""
        r = phase_imbalance(self.total)
        r.pop("L_seq")
        if self.resistance is not None:
            R = self.resistance
            r["R_spread"] = float((R.max() - R.min()) / R.mean())
        return r

    def summary(self) -> pd.DataFrame:
        """Positive-sequence value of each part [mH] and its share of the total."""
        parts = ["magnetising", "harmonic", "slot", "tooth_tip", "end_winding", "leakage", "total"]
        vals = [self.positive(p) for p in parts]
        return pd.DataFrame({"L_positive [mH]": np.array(vals) * 1e3,
                             "share of total [%]": 100 * np.array(vals) / vals[-1]}, index=parts)


def stator_inductances(pattern: WindingPattern, stator: StatorGeometry, end: EndTurn | None = None,
                       g_eff: float | None = None, **resistance) -> StatorInductances:
    """
    All parts of the stator inductance of a winding on a stator, and optionally its resistance.

    Parameters
    ----------
    pattern, stator
    end        : EndTurn | None   End-turn shape; None -> no end winding and no resistance
    g_eff      : float | None     Effective air gap for the air-gap part (see ``airgap_inductance``)
    resistance : keyword arguments for ``phase_resistance`` (a_cond or fill, T, ...);
                 none given -> resistance is None
    """
    m = pattern.m
    ew = end_winding_inductance(pattern, stator, end) if end is not None else np.zeros((m, m))
    R = phase_resistance(pattern, stator, end, **resistance) if (end is not None and resistance) else None
    return StatorInductances(magnetising=airgap_inductance(pattern, stator, "magnetising", g_eff),
                             harmonic=airgap_inductance(pattern, stator, "harmonic", g_eff),
                             slot=slot_leakage_inductance(pattern, stator, "slot"),
                             tooth_tip=slot_leakage_inductance(pattern, stator, "tooth_tip"),
                             end_winding=ew, resistance=R, name=pattern.name)

In [ ]:
#| export
def compare_windings(patterns, stator: StatorGeometry, end: EndTurn | None = None,
                     g_eff: float | None = None, **resistance) -> pd.DataFrame:
    """
    Side-by-side table of candidate windings on the same stator.

    Columns: winding factor, series turns, positive-sequence magnetising, leakage and total
    inductance [mH], harmonic leakage factor σ_δ, self spread and sequence coupling of the total
    [%], ``even fill`` (every used slot position holds the same number of turns, a quick check
    that a hand-made winding can be built), and with resistance arguments the mean resistance [mΩ]
    and its spread [%].

    Parameters
    ----------
    patterns   : iterable of WindingPattern (their names label the rows)
    stator, end, g_eff, resistance : as ``stator_inductances``
    """
    rows = []
    for pat in patterns:
        s = stator_inductances(pat, stator, end, g_eff, **resistance)
        imb = s.imbalance()
        row = {"winding": pat.name, "kw1": pat.check()["kw1"], "N_s": float(pat.series_turns()[0]),
               "L_m [mH]": s.positive("magnetising") * 1e3,
               "sigma_delta": s.positive("harmonic") / s.positive("magnetising"),
               "L_leak [mH]": s.positive("leakage") * 1e3, "L_s [mH]": s.positive() * 1e3,
               "self spread [%]": 100 * imb["self_spread"], "seq coupling [%]": 100 * imb["sequence_coupling"]}
        fill = np.abs(pat.slot_turns).sum(axis=0)
        row["even fill"] = bool(np.allclose(fill[fill > 0], fill[fill > 0].max()))
        if s.resistance is not None:
            row["R [mΩ]"] = float(s.resistance.mean() * 1e3)
            row["R spread [%]"] = 100 * imb["R_spread"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("winding")

---

## Examples

The 36-slot example, all parts:

In [ ]:
stator = StatorGeometry(Q=36, r_bore=60e-3, g=0.8e-3, l_stk=100e-3, h_y=12e-3, l_ew=22e-3,
                        slot=SlotGeometry.semi_closed(b0=2.5e-3, h0=1e-3, h1=1.5e-3, bs=6e-3, hs=18e-3, t=0.5))
pat = WindingPattern.distributed(36, 4, pitch=7, turns=10, parallel_paths=2)
s = stator_inductances(pat, stator, EndTurn(l_ext=12e-3), fill=0.42, T=120)
print(s.summary().round(3))
print(f"zero sequence {s.zero() * 1e3:.4f} mH, R = {np.round(s.resistance * 1e3, 2)} mΩ at 120 °C")

### Comparing windings on one stator

Three windings on the 48-slot stator of the patent example, each with 48 series turns per phase:
- the patent's one-phase-per-layer winding;
- a conventional double layer, short-pitched 10/12;
- a single layer.

The end turns are the same shape for all three, with 8 mm tiers for the three-layer winding. Resistance is at 120 °C with fill 0.45. The table shows what the layer arrangement costs in balance, and what the conventional windings give in exchange (higher $k_{w1}$, so more magnetising inductance per turn).

In [ ]:
def ep2237392(order=("1", "2", "3"), turns=2):
    """EP 2 237 392 B1: 48 slots, 4 poles, one phase per radial layer (phase k in layer order[k])."""
    coils = []
    for ph, (start, sign0) in enumerate([(0, 1), (4, -1), (8, 1)]):
        for pole in range(4):
            s0, sign = start + 12 * pole, sign0 * (-1) ** pole
            pos = order[ph]
            for k in range(6):                                   # concentric: spans 1, 3, ..., 11
                a, b = (s0 + 5 - k) % 48, (s0 + 6 + k) % 48
                if sign < 0:
                    a, b = b, a
                coils.append(Coil(ph, a, pos, b, pos, turns))
    return WindingPattern.custom(48, 4, coils, layout="over_under", n_layers=3,
                                 name="EP2237392")

ep_stator = StatorGeometry(Q=48, r_bore=80e-3, g=0.6e-3, l_stk=150e-3, h_y=15e-3, l_ew=30e-3,
                           slot=SlotGeometry.open(5e-3, 24e-3))

In [ ]:
end = EndTurn(l_ext=12e-3, tier={"1": 16e-3, "2": 8e-3, "3": 0.0})
candidates = [ep2237392(),
              WindingPattern.distributed(48, 4, pitch=10, turns=3),
              WindingPattern.distributed(48, 4, layers=1, turns=6)]
table = compare_windings(candidates, ep_stator, end, fill=0.45, T=120)
table.round(3)

---

## Validation

The parts add up to the total, the positive-sequence value of the sum is the sum of the parts' values, and a balanced winding has zero spread and coupling. For a 3-phase balanced winding, $L_s = L_{aa} - L_{ab}$ and $L_0 = L_{aa} + 2L_{ab}$.

In [ ]:
L = s.total
assert np.allclose(L, airgap_inductance(pat, stator) + slot_leakage_inductance(pat, stator)
                   + end_winding_inductance(pat, stator, EndTurn(l_ext=12e-3)))
assert np.isclose(s.positive(), sum(s.positive(p) for p in ("magnetising", "harmonic", "slot", "tooth_tip", "end_winding")))
assert np.isclose(s.positive(), L[0, 0] - L[0, 1]) and np.isclose(s.zero(), L[0, 0] + 2 * L[0, 1])
imb = s.imbalance()
assert imb["self_spread"] < 1e-12 and imb["sequence_coupling"] < 1e-12 and imb["R_spread"] < 1e-12
print("parts add up; balanced winding has no spread")

---

## Tests

In [ ]:
#| hide
# without end turns: no end winding, no resistance
s0 = stator_inductances(pat, stator)
assert s0.resistance is None and np.allclose(s0.end_winding, 0) and "R_spread" not in s0.imbalance()
# the patent winding is unbalanced in L and R; the conventional ones are not; all fill evenly
t = table
assert t.loc["EP2237392", "self spread [%]"] > 1 and t.loc["EP2237392", "R spread [%]"] > 1
assert t.iloc[1]["self spread [%]"] < 1e-9 and t.iloc[2]["seq coupling [%]"] < 1e-9 and t.iloc[1]["R spread [%]"] < 1e-9
assert t["even fill"].all()
# a winding that overfills some positions is flagged
over = WindingPattern.custom(12, 2, [Coil(0, 0, "top", 6, "bottom", 2), Coil(0, 1, "top", 7, "bottom", 1)], name="uneven")
assert not compare_windings([over], StatorGeometry(Q=12, r_bore=50e-3, g=0.6e-3, l_stk=0.08, h_y=0.01,
                            slot=SlotGeometry.open(5e-3, 20e-3)))["even fill"].iloc[0]
assert np.allclose(t["N_s"], 48)
# compare_windings without resistance has no R columns
assert "R [mΩ]" not in compare_windings(candidates[:1], ep_stator, end).columns

## References

- Pyrhönen, J., Jokinen, T., Hrabovcová, V. (2014). *Design of Rotating Electrical Machines* (2nd ed.). Wiley, ch. 4 — synchronous inductance as magnetising plus leakage.
- Krause, P.C., Wasynczuk, O., Sudhoff, S.D., Pekarek, S. (2013). *Analysis of Electric Machinery and Drive Systems* (3rd ed.). Wiley-IEEE Press — phase, sequence and dq inductances.